In [ ]:
import pickle
from tqdm import tqdm
import numpy as np
from gymnasium_env.envs.auxiliary.utils import decode, encode

from gymnasium_env.envs.blokus_piece import BlokusPieceManager

In [ ]:
def compute_neighborhood_positions():
    cnt = 0
    neighbor_idx, neighbor_pos = {}, []
    for i in range(-2, 1):
        for j in range(1, 4 - (-i // 2)):
            neighbor_idx[i, j] = cnt
            neighbor_pos.append((i, j))
            # print(i, j, cnt)
            cnt += 1
    for i in range(1, 3):
        for j in range(-2, 5 - i):
            neighbor_idx[i, j] = cnt
            neighbor_pos.append((i, j))
            # print(i, j, cnt)
            cnt += 1
    for i in range(-1, 2):
        neighbor_idx[3, i] = cnt
        neighbor_pos.append((3, i))
        # print(-3, i, cnt)
        cnt += 1
    return neighbor_idx, neighbor_pos

In [ ]:
def compute_neighborhood_actions():
    to_idx, to_pos = compute_neighborhood_positions()
    compute_actions = [[] for _ in range(1 << 22)]

    for i in tqdm(range(91)):
        piece = BlokusPieceManager().get_piece(piece_id=i)
        for h, w in piece.body:
            good_cover = True
            ids = []
            for xx, yy in piece.body:
                x, y = xx - h, yy - w
                if not ((x, y) in to_idx) and not (x == 0 and y == 0):
                    good_cover = False
                    break
                elif x != 0 or y != 0:
                    ids.append(to_idx[(x, y)])
                pass

            if not good_cover:
                continue

            for j in range(1 << 22):
                all_ids = True
                for k in ids:
                    if not (j & (1 << k)):
                        all_ids = False
                if all_ids:
                    compute_actions[j].append(encode(-h, -w, i))
                    
    compute_actions = np.array([np.array(actions, dtype=np.int16) for actions in compute_actions], dtype=object)
    return compute_actions

In [ ]:
directory = 'pre_neighbors'
path = f'{directory}/compute_actions.pkl'

neighbor_idx, neighbor_pos = compute_neighborhood_positions()

compute_actions = compute_neighborhood_actions()

# data = {
#     'neighbor_idx': neighbor_idx,
#     'neighbor_pos': neighbor_pos,
#     'compute_actions': compute_actions
# }

In [ ]:
path = f'{directory}/compute_actions.npy'

# np.save(path, compute_actions)
np.save(f'{directory}/neighbor_pos.npy', neighbor_pos)
print("Done")

In [ ]:
# with open('pre_neighbors/compute_actions.pkl', 'rb') as f:
#     data = pickle.load(f)
#     compute_actions = data['compute_actions']
#     neighbor_idx = data['neighbor_idx']
#     neighbor_pos = data['neighbor_pos']

# print(compute_actions)
# print(neighbor_idx)
# print(neighbor_pos)